In [1]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

# 1. Hardware device selection (Apple Silicon MPS or CPU)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using compute device: {device}")

# 2. Load the Oracle dataset generated in Lab 2
data_path = Path("../data/oracle_dataset.parquet")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path.resolve()}. Please run Lab 2 first!")

df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} samples from {data_path.name}")
print(f"Target Distribution: Small={sum(df['route_to_frontier'] == 0)}, Frontier={sum(df['route_to_frontier'] == 1)}")

# 3. Load the Bi-Encoder to generate the 384-dim dense semantic embeddings
print("\nLoading Bi-Encoder: all-MiniLM-L6-v2...")
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

print("Setup completed successfully!")

/Users/mathew/Desktop/routellm-gateway/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1104 22:41:18.170000 60395 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Using compute device: mps
Loaded 600 samples from oracle_dataset.parquet
Target Distribution: Small=270, Frontier=330

Loading Bi-Encoder: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6045.02it/s]


Setup completed successfully!


In [2]:
import re

def compute_nesting_depth(text: str) -> int:
    """Computes the maximum nesting level of JSON/curly braces."""
    max_depth = current_depth = 0
    for char in text:
        if char == "{":
            current_depth += 1
            max_depth = max(max_depth, current_depth)
        elif char == "}":
            current_depth = max(0, current_depth - 1)
    return max_depth

def extract_structural_features(prompt: str) -> list:
    """
    Extracts 12 structural complexity signals from the raw prompt text:
    1. token_count (approximate)
    2. char_length
    3. schema_nesting_depth
    4. total_brace_count
    5. json_constraint_flag
    6. code_syntax_count
    7. concurrency_flag
    8. sql_window_function_flag
    9. math_symbol_count
    10. multi_constraint_count
    11. question_mark_count (interrogative / factual signal)
    12. number_density (ratio of numeric characters)
    """
    p_lower = prompt.lower()
    words = prompt.split()
    n_chars = max(len(prompt), 1)
    
    # Structural signals
    token_count = len(words)
    char_length = len(prompt)
    nesting_depth = compute_nesting_depth(prompt)
    brace_count = prompt.count("{") + prompt.count("}")
    json_flag = int("json" in p_lower or "schema" in p_lower or "raw" in p_lower or "no markdown" in p_lower)
    
    code_words = ["def ", "class ", "import ", "return ", "async ", "->", "()", ";"]
    code_count = sum(p_lower.count(w) for w in code_words)
    
    conc_flag = int(any(w in p_lower for w in ["thread", "lock", "async", "backoff", "race"]))
    sql_flag = int(any(w in p_lower for w in ["window", "dense_rank", "row_number", "partition"]))
    
    math_signals = ["+", "*", "/", "%", ">", "<", "=", "efficiency", "pallets", "req/s"]
    math_count = sum(prompt.count(s) for s in math_signals)
    
    constraint_words = ["must", "cannot", "only", "strictly", "rules:", "ensure", "constraint:"]
    constraint_count = sum(p_lower.count(w) for w in constraint_words)
    
    q_count = prompt.count("?")
    num_density = sum(c.isdigit() for c in prompt) / n_chars
    
    return [
        token_count, char_length, nesting_depth, brace_count, json_flag,
        code_count, conc_flag, sql_flag, math_count, constraint_count,
        q_count, num_density
    ]

# Feature names for explainability in later ablation analysis
structural_feature_names = [
    "token_count", "char_length", "schema_nesting_depth", "total_brace_count",
    "json_constraint_flag", "code_syntax_count", "concurrency_flag", 
    "sql_window_flag", "math_symbol_count", "multi_constraint_count",
    "question_mark_count", "number_density"
]

print(f"Registered {len(structural_feature_names)} structural features:")
for i, name in enumerate(structural_feature_names, 1):
    print(f"  {i:2d}. {name}")

Registered 12 structural features:
   1. token_count
   2. char_length
   3. schema_nesting_depth
   4. total_brace_count
   5. json_constraint_flag
   6. code_syntax_count
   7. concurrency_flag
   8. sql_window_flag
   9. math_symbol_count
  10. multi_constraint_count
  11. question_mark_count
  12. number_density


In [3]:
from sklearn.preprocessing import StandardScaler

print("1. Extracting 384-dimensional dense semantic embeddings...")
prompts = df["prompt"].tolist()
X_dense = bi_encoder.encode(prompts, convert_to_numpy=True, normalize_embeddings=True)
print(f"   Dense feature matrix shape: {X_dense.shape}")

print("2. Extracting 12 structural complexity features...")
raw_struct_features = [extract_structural_features(p) for p in prompts]
X_struct_raw = np.array(raw_struct_features, dtype=np.float32)

# Scale structural features so they are on the same magnitude as normalized embeddings
scaler = StandardScaler()
X_struct_scaled = scaler.fit_transform(X_struct_raw)
print(f"   Structural feature matrix shape: {X_struct_scaled.shape}")

print("3. Concatenating into final 396-dimensional hybrid feature matrix...")
X_hybrid = np.hstack([X_dense, X_struct_scaled])
y = df["route_to_frontier"].values

print(f"\nFinal Feature Matrix (X) shape: {X_hybrid.shape}")
print(f"Target Labels (y) shape:         {y.shape}")
print(f"Class Balance: {np.mean(y == 0)*100:.1f}% Small (0) vs {np.mean(y == 1)*100:.1f}% Frontier (1)")

1. Extracting 384-dimensional dense semantic embeddings...
   Dense feature matrix shape: (600, 384)
2. Extracting 12 structural complexity features...
   Structural feature matrix shape: (600, 12)
3. Concatenating into final 396-dimensional hybrid feature matrix...

Final Feature Matrix (X) shape: (600, 396)
Target Labels (y) shape:         (600,)
Class Balance: 45.0% Small (0) vs 55.0% Frontier (1)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, classification_report

# 1. Leak-Free Train-Test Split (80% Train, 20% Test, stratified by class balance)
# Stratify ensures both train and test have the exact same 45% Small / 55% Frontier ratio
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=y)

print(f"Training Samples: {len(train_idx)} | Testing Samples: {len(test_idx)}")

# 2. Extract split features
X_dense_train, X_dense_test = X_dense[train_idx], X_dense[test_idx]
X_struct_train_raw, X_struct_test_raw = X_struct_raw[train_idx], X_struct_raw[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# 3. FIT the scaler ONLY on training data, then TRANSFORM test data (Zero Data Leakage!)
struct_scaler = StandardScaler()
X_struct_train = struct_scaler.fit_transform(X_struct_train_raw)
X_struct_test = struct_scaler.transform(X_struct_test_raw)  # Notice: transform only!

# 4. Assemble the final 396-dim hybrid matrices
X_train_hybrid = np.hstack([X_dense_train, X_struct_train])
X_test_hybrid = np.hstack([X_dense_test, X_struct_test])

# 5. Build the Calibrated Router:
# Base estimator: L2 Logistic Regression
base_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

# Wrap in 5-fold Platt Scaling (method='sigmoid') for calibrated probabilities
calibrated_router = CalibratedClassifierCV(estimator=base_lr, method="sigmoid", cv=5)
calibrated_router.fit(X_train_hybrid, y_train)

# 6. Evaluate on Unseen Test Traffic
test_preds = calibrated_router.predict(X_test_hybrid)
test_probs = calibrated_router.predict_proba(X_test_hybrid)[:, 1]

acc = accuracy_score(y_test, test_preds)
auc = roc_auc_score(y_test, test_probs)
brier = brier_score_loss(y_test, test_probs)

print("=" * 60)
print("LAB 3: CALIBRATED HYBRID ROUTER EVALUATION (TEST SET)")
print("=" * 60)
print(f"Test Accuracy:           {acc * 100:.2f}%")
print(f"ROC-AUC Score:           {auc:.4f}  (1.0 is perfect discrimination)")
print(f"Brier Score (Loss):      {brier:.4f}  (0.0 is perfect probability calibration)")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=["Route to Small (0)", "Route to Frontier (1)"]))

Training Samples: 480 | Testing Samples: 120
LAB 3: CALIBRATED HYBRID ROUTER EVALUATION (TEST SET)
Test Accuracy:           100.00%
ROC-AUC Score:           1.0000  (1.0 is perfect discrimination)
Brier Score (Loss):      0.0047  (0.0 is perfect probability calibration)

Classification Report:
                       precision    recall  f1-score   support

   Route to Small (0)       1.00      1.00      1.00        54
Route to Frontier (1)       1.00      1.00      1.00        66

             accuracy                           1.00       120
            macro avg       1.00      1.00      1.00       120
         weighted avg       1.00      1.00      1.00       120



In [5]:
# To inspect the base linear weights, fit a standalone LogisticRegression on the training set
interpret_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
interpret_lr.fit(X_train_hybrid, y_train)

# The first 384 weights belong to dense embeddings, the last 12 belong to structural features
structural_weights = interpret_lr.coef_[0][384:]

# Create an explainability DataFrame
coef_df = pd.DataFrame({
    "Feature": structural_feature_names,
    "Learned Weight (w)": structural_weights,
    "Routing Impact": [
        "Escalate to Frontier (+)" if w > 0.05 else ("Route to Small (-)" if w < -0.05 else "Neutral (~0)")
        for w in structural_weights
    ]
}).sort_values(by="Learned Weight (w)", ascending=False).reset_index(drop=True)

print("=" * 65)
print("LAB 3: LEARNED WEIGHTS FOR STRUCTURAL COMPLEXITY FEATURES")
print("=" * 65)
coef_df

LAB 3: LEARNED WEIGHTS FOR STRUCTURAL COMPLEXITY FEATURES


,Feature,Learned Weight (w),Routing Impact
0,multi_constraint_count,2.705371,Escalate to Frontier (+)
1,char_length,1.671332,Escalate to Frontier (+)
2,number_density,1.182072,Escalate to Frontier (+)
3,code_syntax_count,1.175048,Escalate to Frontier (+)
4,sql_window_flag,0.890718,Escalate to Frontier (+)
5,token_count,0.839510,Escalate to Frontier (+)
6,concurrency_flag,0.816941,Escalate to Frontier (+)
7,schema_nesting_depth,0.761435,Escalate to Frontier (+)
8,json_constraint_flag,0.736877,Escalate to Frontier (+)
9,total_brace_count,0.681774,Escalate to Frontier (+)


In [6]:
# Helper function to train and evaluate a calibrated model on specific features
def evaluate_feature_subset(X_tr, X_te, name):
    base = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    clf = CalibratedClassifierCV(estimator=base, method="sigmoid", cv=5)
    clf.fit(X_tr, y_train)
    
    preds = clf.predict(X_te)
    probs = clf.predict_proba(X_te)[:, 1]
    
    return {
        "Feature Set": name,
        "Dimensions": X_tr.shape[1],
        "Accuracy": f"{accuracy_score(y_test, preds) * 100:.2f}%",
        "ROC-AUC": f"{roc_auc_score(y_test, probs):.4f}",
        "Brier Score (Loss)": f"{brier_score_loss(y_test, probs):.4f}"
    }

# 1. Evaluate Model 1: Dense Embeddings Only (384-dim)
res_dense = evaluate_feature_subset(X_dense_train, X_dense_test, "Dense Embeddings Only (MiniLM-384)")

# 2. Evaluate Model 2: Structural Features Only (12-dim)
res_struct = evaluate_feature_subset(X_struct_train, X_struct_test, "Structural Complexity Only (12 Signals)")

# 3. Evaluate Model 3: Hybrid Architecture (396-dim)
res_hybrid = evaluate_feature_subset(X_train_hybrid, X_test_hybrid, "Hybrid Router (384 Dense + 12 Structural)")

# Build the Comparison Table
ablation_df = pd.DataFrame([res_dense, res_struct, res_hybrid])

print("=" * 75)
print("LAB 3: FEATURE ABLATION STUDY COMPARISON TABLE")
print("=" * 75)
ablation_df

LAB 3: FEATURE ABLATION STUDY COMPARISON TABLE


,Feature Set,Dimensions,Accuracy,ROC-AUC,Brier Score (Loss)
0,Dense Embeddings Only (MiniLM-384),384,100.00%,1.0000,0.0039
1,Structural Complexity Only (12 Signals),12,100.00%,1.0000,0.0102
2,Hybrid Router (384 Dense + 12 Structural),396,100.00%,1.0000,0.0047


In [7]:
coef_df["Odds Multiplier (e^w)"] = np.exp(coef_df["Learned Weight (w)"])
coef_df[["Feature", "Learned Weight (w)", "Odds Multiplier (e^w)"]]

,Feature,Learned Weight (w),Odds Multiplier (e^w)
0,multi_constraint_count,2.705371,14.959861
1,char_length,1.671332,5.319248
2,number_density,1.182072,3.261126
3,code_syntax_count,1.175048,3.238298
4,sql_window_flag,0.890718,2.436879
5,token_count,0.839510,2.315233
6,concurrency_flag,0.816941,2.263565
7,schema_nesting_depth,0.761435,2.141346
8,json_constraint_flag,0.736877,2.089399
9,total_brace_count,0.681774,1.977383


In [8]:
import joblib

# 1. Define artifacts directory
artifacts_dir = Path("../backend/app/router/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

# 2. Save the trained Calibrated Classifier & the Feature Scaler
model_path = artifacts_dir / "calibrated_router.joblib"
scaler_path = artifacts_dir / "struct_scaler.joblib"

joblib.dump(calibrated_router, model_path)
joblib.dump(struct_scaler, scaler_path)

print("=" * 65)
print("LAB 3 COMPLETE: ROUTER ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 65)
print(f"Calibrated Model saved to: {model_path.resolve()}")
print(f"Feature Scaler saved to:   {scaler_path.resolve()}")
print("=" * 65)

# Verify loading works cleanly
loaded_router = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)
print("Verification: Artifacts reloaded from disk without errors!")

LAB 3 COMPLETE: ROUTER ARTIFACTS SAVED SUCCESSFULLY
Calibrated Model saved to: /Users/mathew/Desktop/routellm-gateway/backend/app/router/artifacts/calibrated_router.joblib
Feature Scaler saved to:   /Users/mathew/Desktop/routellm-gateway/backend/app/router/artifacts/struct_scaler.joblib
Verification: Artifacts reloaded from disk without errors!
